# Note Agent Demo

Демо показывает двух writer-пользователей User1 и User2, режимы `none` и `edit`, раздельную memory, изоляцию заметок и сценарий с бостонским пирогом.

In [1]:
from pathlib import Path
import os
import sys

project_root = Path.cwd()
if not (project_root / "bot.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

demo_db = project_root / "data" / "demo_notebook.sqlite"
if demo_db.exists():
    demo_db.unlink()

os.environ["NOTES_DB"] = str(demo_db)

project_root


WindowsPath('g:/Note-Agent')

In [2]:
from bot import User, ConsoleBot, classify_intent, classify_reply, classify_reply_by_rules
from config import get_settings

settings = get_settings()

print("OpenRouter model:", settings.openrouter_model)
print("OpenRouter classifier model:", settings.openrouter_classifier_model)
print("OpenRouter base URL:", settings.openrouter_base_url)
print("API key loaded:", bool(settings.openrouter_api_key))


OpenRouter model: qwen/qwen3.7-plus
OpenRouter classifier model: google/gemma-4-31b-it
OpenRouter base URL: https://openrouter.ai/api/v1
API key loaded: True


In [3]:
user1 = User("user1")
user2 = User("user2")

print(f"User1: {user1.user_id}, role={user1.role}")
print(f"User2: {user2.user_id}, role={user2.role}")


User1: user1, role=writer
User2: user2, role=writer


## User2, режим none

User2 работает в режиме `none`: general LLM не видит заметки. При этом обычные команды заметок всё равно идут через MCP.

In [4]:
bot_user2 = ConsoleBot(user2, notes_access_mode="none")
await bot_user2.initialize()

print("Текущий пользователь:", bot_user2.user.user_id)
print("Режим заметок:", bot_user2.notes_access_mode)
print("Memory User2 до вопроса:", bot_user2.memory)

print(await bot_user2.handle("добавь заметку: список дел User2: проверить почту и купить чай"))
print(await bot_user2.handle("покажи мои заметки"))

answer = await bot_user2.handle("В каких блюдах используют барбарис?")
print(answer)

print("Memory User2 после general-вопроса:", bot_user2.memory)


Текущий пользователь: user2
Режим заметок: none
Memory User2 до вопроса: []
Заметка сохранена. id=1
1. список дел User2: проверить почту и купить чай (2026-06-27T11:01:25.543236+00:00)
Барбарис (чаще всего в сушеном виде) ценится в кулинарии за свою яркую кислинку, терпкость и легкий фруктово-хвойный аромат. Он отлично балансирует жирные и тяжелые блюда. 

Вот основные категории блюд, где он используется:

**1. Блюда из круп и мяса (Восточная и Кавказская кухня)**
*   **Плов:** Это самое классическое применение. Сушеные ягоды добавляют узбекскому и другим видам плова пикантную кислинку, которая оттеняет вкус жирного мяса, а также красивый внешний вид.
*   **Тушеное мясо:** Барбарис добавляют к баранине, говядине и дичи. Кислота помогает смягчить мясные волокна и делает вкус блюда более насыщенным.
*   **Шашлыки и кебабы:** Иногда ягоды нанизывают на шампур вместе с мясом или добавляют в маринад.

**2. Напитки**
*   **Чай:** Сушеные ягоды или листья барбариса добавляют в черный или зеле

## Переключение на User1, режим edit

Создаём нового агента для User1. У него новая пустая memory: переписка User2 не попадает к User1.

In [5]:
bot_user1 = ConsoleBot(user1, notes_access_mode="edit")
await bot_user1.initialize()

print("Текущий пользователь:", bot_user1.user.user_id)
print("Режим заметок:", bot_user1.notes_access_mode)
print("Memory User1 сразу после переключения:", bot_user1.memory)
print("Memory User2 осталась в старом объекте bot_user2:", bot_user2.memory)


Текущий пользователь: user1
Режим заметок: edit
Memory User1 сразу после переключения: []
Memory User2 осталась в старом объекте bot_user2: [('human', 'В каких блюдах используют барбарис?'), ('ai', 'Барбарис (чаще всего в сушеном виде) ценится в кулинарии за свою яркую кислинку, терпкость и легкий фруктово-хвойный аромат. Он отлично балансирует жирные и тяжелые блюда. \n\nВот основные категории блюд, где он используется:\n\n**1. Блюда из круп и мяса (Восточная и Кавказская кухня)**\n*   **Плов:** Это самое классическое применение. Сушеные ягоды добавляют узбекскому и другим видам плова пикантную кислинку, которая оттеняет вкус жирного мяса, а также красивый внешний вид.\n*   **Тушеное мясо:** Барбарис добавляют к баранине, говядине и дичи. Кислота помогает смягчить мясные волокна и делает вкус блюда более насыщенным.\n*   **Шашлыки и кебабы:** Иногда ягоды нанизывают на шампур вместе с мясом или добавляют в маринад.\n\n**2. Напитки**\n*   **Чай:** Сушеные ягоды или листья барбариса доб

In [6]:
examples = [
    "добавь заметку: купить 2 пакета барбариса",
    "покажи мои заметки",
    "удали заметку 1",
    "каких ингредиентов не хватает для бостонского пирога?",
]

for text in examples:
    intent = await classify_intent(text, bot_user1.classifier_llm)
    print(text, "->", intent)


добавь заметку: купить 2 пакета барбариса -> Intent(kind='add_note', note_text='купить 2 пакета барбариса', note_id=None)
покажи мои заметки -> Intent(kind='list_notes', note_text=None, note_id=None)
удали заметку 1 -> Intent(kind='delete_note', note_text=None, note_id=1)
каких ингредиентов не хватает для бостонского пирога? -> Intent(kind='chat', note_text=None, note_id=None)


In [7]:
shopping_note = (
    "добавь заметку: список покупок для бостонского пирога: "
    "1л молока, 10 яиц C0, 1 кг муки, оливковое масло 700 ml, 200грамм сыровяленой колбасы"
)

print(await bot_user1.handle(shopping_note))
print(await bot_user1.handle("покажи мои заметки"))


Заметка сохранена. id=2
2. список покупок для бостонского пирога: 1л молока, 10 яиц C0, 1 кг муки, оливковое масло 700 ml, 200грамм сыровяленой колбасы (2026-06-27T11:02:57.297250+00:00)


In [8]:
question = (
    "Я хочу приготовить бостонский пирог. "
    "Каких ингредиентов не хватает в моём списке покупок? "
    "Если нужно, предложи добавить недостающие ингредиенты в заметку."
)

answer = await bot_user1.handle(question)
print(answer)
print("\nPending action User1:", bot_user1.pending_action)
print("\nMemory User1 после вопроса:", bot_user1.memory)


Для классического бостонского крем-пирога (Boston Cream Pie) в вашем списке не хватает: сахара, сливочного масла, разрыхлителя, ванилина, шоколада, кукурузного крахмала и соли. Также обратите внимание, что сыровяленая колбаса обычно не используется в этом десерте, но я оставил её в списке. Подтвердите, пожалуйста, если вы хотите добавить недостающие ингредиенты в заметку.

Предложенное действие: обновить заметку id=2
Причина: Добавление недостающих ингредиентов для приготовления классического бостонского крем-пирога
Напиши `да`, если нужно выполнить это изменение.

Pending action User1: {'type': 'update_note', 'id': 2, 'text': 'список покупок для бостонского пирога: 1л молока, 10 яиц C0, 1 кг муки, оливковое масло 700 ml, 200грамм сыровяленой колбасы, сахар, сливочное масло, разрыхлитель, ванилин, шоколад, кукурузный крахмал, соль', 'reason': 'Добавление недостающих ингредиентов для приготовления классического бостонского крем-пирога'}

Memory User1 после вопроса: [('human', 'Я хочу пр

In [9]:
reply_text = "yeah, let's cook it!" #не входит в obvious_confirm

if bot_user1.pending_action:
    rule_result = classify_reply_by_rules(reply_text)

    if rule_result is None:
        print("classify_reply_by_rules не нашла очевидное да/нет, вызываем LLM-классификатор.")
        reply = await classify_reply(
            reply_text,
            bot_user1.pending_action_context(),
            bot_user1.classifier_llm,
        )
        print("Reply intent:", reply["intent"])
        print("Reply confidence:", reply["confidence"])
        print("Needs clarification:", reply["needs_clarification"])

        if bot_user1.can_execute_reply(reply):
            print(await bot_user1.execute_pending_action())
        elif reply["intent"] == "reject" and reply["confidence"] >= 0.75:
            bot_user1.pending_action = None
            print("Ок, изменение заметок отменено.")
        else:
            print(bot_user1.clarification_for_reply(reply))
    else:
        print("classify_reply_by_rules обработала ответ без LLM:", rule_result)
        print(await bot_user1.handle(reply_text))
else:
    print("LLM не предложила действие для подтверждения.")

print("\nФинальное содержимое заметок User1:")
print(await bot_user1.handle("покажи мои заметки"))


classify_reply_by_rules не нашла очевидное да/нет, вызываем LLM-классификатор.
Reply intent: confirm
Reply confidence: 0.95
Needs clarification: False
Готово. Заметка после редактирования:
2. список покупок для бостонского пирога: 1л молока, 10 яиц C0, 1 кг муки, оливковое масло 700 ml, 200грамм сыровяленой колбасы, сахар, сливочное масло, разрыхлитель, ванилин, шоколад, кукурузный крахмал, соль

Финальное содержимое заметок User1:
2. список покупок для бостонского пирога: 1л молока, 10 яиц C0, 1 кг муки, оливковое масло 700 ml, 200грамм сыровяленой колбасы, сахар, сливочное масло, разрыхлитель, ванилин, шоколад, кукурузный крахмал, соль (2026-06-27T11:02:57.297250+00:00)


## Новый агент User2

Создаём новый объект агента для User2. У нового объекта memory снова пустая, заметки User2 сохранены, а заметки User1 ему не видны.

In [10]:
bot_user2_again = ConsoleBot(user2, notes_access_mode="none")
await bot_user2_again.initialize()

print("Текущий пользователь:", bot_user2_again.user.user_id)
print("Режим заметок:", bot_user2_again.notes_access_mode)
print("Memory нового User2-агента:", bot_user2_again.memory)
print("Заметки User2:")
print(await bot_user2_again.handle("покажи мои заметки"))


Текущий пользователь: user2
Режим заметок: none
Memory нового User2-агента: []
Заметки User2:
1. список дел User2: проверить почту и купить чай (2026-06-27T11:01:25.543236+00:00)


In [11]:
bot_user1_again = ConsoleBot(user1, notes_access_mode="none")
await bot_user1_again.initialize()

print("Текущий пользователь:", bot_user1_again.user.user_id)
print("Режим заметок:", bot_user1_again.notes_access_mode)
print("Memory нового User1-агента:", bot_user1_again.memory)
print("Заметки User1:")
print(await bot_user1_again.handle("покажи мои заметки"))

Текущий пользователь: user1
Режим заметок: none
Memory нового User1-агента: []
Заметки User1:
2. список покупок для бостонского пирога: 1л молока, 10 яиц C0, 1 кг муки, оливковое масло 700 ml, 200грамм сыровяленой колбасы, сахар, сливочное масло, разрыхлитель, ванилин, шоколад, кукурузный крахмал, соль (2026-06-27T11:02:57.297250+00:00)
